In [1]:
from sklearn.model_selection import train_test_split
import boto3
import pandas as pd

sm_boto3 = boto3.client('sagemaker')
# Get region from boto3 session
session = boto3.Session()
region = session.region_name
bucket = "mobile-sage-maker-bucket"
print(f"Using region: {region}")
print(f"Using bucket: {bucket}")

Using region: us-east-1
Using bucket: mobile-sage-maker-bucket


In [4]:
df=pd.read_csv("data\mob_price_classification_train.csv")
df.head()

,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,...,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range
0,842,0,2.2,0,1,0,7,0.6,188,2,...,20,756,2549,9,7,19,0,0,1,1
1,1021,1,0.5,1,0,1,53,0.7,136,3,...,905,1988,2631,17,3,7,1,1,0,2
2,563,1,0.5,1,2,1,41,0.9,145,5,...,1263,1716,2603,11,2,9,1,1,0,2
3,615,1,2.5,0,0,0,10,0.8,131,6,...,1216,1786,2769,16,8,11,1,0,0,2
4,1821,1,1.2,0,13,1,44,0.6,141,2,...,1208,1212,1411,8,2,15,1,1,0,1


In [5]:
df.shape

(2000, 21)

In [6]:
df.isnull().sum()

battery_power    0
blue             0
clock_speed      0
dual_sim         0
fc               0
four_g           0
int_memory       0
m_dep            0
mobile_wt        0
n_cores          0
pc               0
px_height        0
px_width         0
ram              0
sc_h             0
sc_w             0
talk_time        0
three_g          0
touch_screen     0
wifi             0
price_range      0
dtype: int64

In [7]:
df.price_range.value_counts()

price_range
1    500
2    500
3    500
0    500
Name: count, dtype: int64

In [8]:
df.columns

Index(['battery_power', 'blue', 'clock_speed', 'dual_sim', 'fc', 'four_g',
       'int_memory', 'm_dep', 'mobile_wt', 'n_cores', 'pc', 'px_height',
       'px_width', 'ram', 'sc_h', 'sc_w', 'talk_time', 'three_g',
       'touch_screen', 'wifi', 'price_range'],
      dtype='object')

In [9]:
features = df.columns[:-1]
label = df.columns[-1]

In [10]:
x=df.drop(label,axis=1)
y=df[label]

In [11]:
X_train,X_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [12]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(1600, 20)
(400, 20)
(1600,)
(400,)


In [13]:
train_X = pd.DataFrame(X_train)
train_X[label] = y_train

test_X = pd.DataFrame(X_test)
test_X[label] = y_test

In [14]:
test_X

,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,...,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range
1860,1646,0,2.5,0,3,1,25,0.6,200,2,...,211,1608,686,8,6,11,1,1,0,0
353,1182,0,0.5,0,7,1,8,0.5,138,8,...,275,986,2563,19,17,19,1,0,0,2
1333,1972,0,2.9,0,9,0,14,0.4,196,7,...,293,952,1316,8,1,8,1,1,0,1
905,989,1,2.0,0,4,0,17,0.2,166,3,...,256,1394,3892,18,7,19,1,1,0,3
1289,615,1,0.5,1,7,0,58,0.5,130,5,...,1021,1958,1906,14,5,5,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
965,1379,0,0.5,1,1,0,19,0.3,134,8,...,387,671,3912,11,2,19,0,1,1,3
1284,991,0,2.0,0,2,1,12,0.3,158,5,...,1209,1678,2014,11,9,10,1,0,0,2
1739,1044,0,1.8,0,4,1,12,0.7,104,6,...,1230,1263,1794,18,7,19,1,1,1,1
261,728,0,2.7,1,0,0,25,0.2,88,4,...,526,1529,2039,5,1,12,1,1,1,1


In [15]:
train_X.to_csv("train.csv", index=False)
test_X.to_csv("test.csv", index=False)

In [16]:
bucket

'mobile-sage-maker-bucket'

In [17]:
## Upload the training and test data to S3
import os
import boto3

sk_prefix = "sagemaker/mobile-price-classification/sklearncontainer"

# Make this cell independent of execution order.
session = globals().get("session", boto3.Session())
bucket = globals().get("bucket", "mobile-sage-maker-bucket")
s3 = session.client("s3")

train_key = f"{sk_prefix}/train.csv"
test_key = f"{sk_prefix}/test.csv"

if not os.path.exists("train.csv") or not os.path.exists("test.csv"):
    raise FileNotFoundError("train.csv/test.csv not found. Run the train-test split and CSV export cells first.")

s3.upload_file("train.csv", bucket, train_key)
s3.upload_file("test.csv", bucket, test_key)

train_path = f"s3://{bucket}/{train_key}"
test_path = f"s3://{bucket}/{test_key}"

print(train_path)
print(test_path)

s3://mobile-sage-maker-bucket/sagemaker/mobile-price-classification/sklearncontainer/train.csv
s3://mobile-sage-maker-bucket/sagemaker/mobile-price-classification/sklearncontainer/test.csv


In [42]:
%%writefile script.py

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score,
                            classification_report,
                            confusion_matrix,
                            precision_score)
import sklearn
import joblib
import boto3
import pathlib
from io import StringIO 
import argparse
import os
import pandas as pd
import numpy as np

def model_fn(model_dir):
    model=joblib.load(os.path.join(model_dir, "model.joblib"))


if __name__ == "__main__":

    print("[info] Extracting arguments.")
    parser=argparse.ArgumentParser()

    ## hyperparameters
    parser.add_argument("--n-estimators", type=int, default=100)
    parser.add_argument("--max-depth", type=int, default=5)

    ## data, model directories
    parser.add_argument("--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR"))
    parser.add_argument("--train", type=str, default=os.environ.get("SM_CHANNEL_TRAIN"))
    parser.add_argument("--test", type=str, default=os.environ.get("SM_CHANNEL_TEST"))

    parser.add_argument("--train-file", type=str, default="train.csv")
    parser.add_argument("--test-file", type=str, default="test.csv")

    args, _ = parser.parse_known_args()

    print("Scikit-learn version:", sklearn.__version__)
    print("Joblib version:", joblib.__version__)

    print("[info] Reading training and test data.")
    print()

    train_data = pd.read_csv(os.path.join(args.train, args.train_file))
    test_data = pd.read_csv(os.path.join(args.test, args.test_file))

    features = train_data.columns[:-1]
    label = train_data.columns[-1]

    print("Building training and test datasets.")
    print()
    X_train = train_data[features]
    y_train = train_data[label]
    X_test = test_data[features]
    y_test = test_data[label]

    print("Column order")
    print()

    print(X_train.columns)
    print(X_test.columns)
    print()

    print("Label column is:", label)
    print()

    print("Data shapes:")
    print("Training data shape:", X_train.shape)
    print("Test data shape:", X_test.shape)
    print()

    print("[info] Training model.")
    print()

    model = RandomForestClassifier(
                    n_estimators=args.n_estimators,
                    max_depth=args.max_depth,
                    random_state=42,
                    verbose=1,
                    n_jobs=-1
            )
    model.fit(X_train, y_train)
    print()

    model_path = os.path.join(args.model_dir, "model.joblib")
    joblib.dump(model, model_path)

    print("Model saved to:", model_path)

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    test_report = classification_report(y_test, y_pred)
    conf_matrix = confusion_matrix(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="weighted")

    print("Test Accuracy:", acc)
    print("Test Precision:", precision)
    print("Classification Report:")
    print(test_report)
    print("Confusion Matrix:")
    print(conf_matrix)
    print()

Writing script.py


In [18]:
## AWS SageMaker Entry Point to Execute the Training Script

from sagemaker.sklearn.estimator import SKLearn

FRAMEWORK_VERSION = "0.23-1"
ROLE_ARN = "arn:aws:iam::889993502497:role/sageMakerAccess"
INSTANCE_TYPE_CANDIDATES = ["ml.m5.large", "ml.m5.xlarge", "ml.c5.xlarge"]


def build_estimator(instance_type: str) -> SKLearn:
    return SKLearn(
        entry_point="script.py",
        role=ROLE_ARN,
        instance_count=1,
        instance_type=instance_type,
        framework_version=FRAMEWORK_VERSION,
        base_job_name="sklearn-mobile-price-classification",
        hyperparameters={
            "n-estimators": 100,
            "random-state": 42,
            "max-depth": 5,
        },
        use_spot_instances=False,
        py_version="py3",
    )

sagemaker.config INFO - Not applying SDK defaults from location: C:\ProgramData\sagemaker\sagemaker\config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: C:\Users\yshan\AppData\Local\sagemaker\sagemaker\config.yaml


In [ ]:
last_error = None
sklearn_estimator = None

for instance_type in INSTANCE_TYPE_CANDIDATES:
    try:
        print(f"Starting training on {instance_type} (On-Demand)...")
        sklearn_estimator = build_estimator(instance_type)
        sklearn_estimator.fit({"train": train_path, "test": test_path}, wait=True)
        print(f"Training succeeded on {instance_type}")
        break
    except Exception as exc:
        last_error = exc
        print(f"Training failed on {instance_type}: {exc}")

if sklearn_estimator is None:
    raise RuntimeError("All candidate instance types failed.") from last_error

Starting training on ml.m5.large (On-Demand)...


INFO:sagemaker:Creating training-job with name: sklearn-mobile-price-classification-2026-04-15-16-40-05-113


2026-04-15 16:40:13 Starting - Starting the training job...
2026-04-15 16:40:36 Starting - Preparing the instances for training...
2026-04-15 16:40:58 Downloading - Downloading input data...
2026-04-15 16:41:28 Downloading - Downloading the training image..2026-04-15 16:42:18,018 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
2026-04-15 16:42:18,023 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-04-15 16:42:18,073 sagemaker_sklearn_container.training INFO     Invoking user training script.
2026-04-15 16:42:18,297 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-04-15 16:42:18,310 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-04-15 16:42:18,324 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-04-15 16:42:18,334 sagemaker-training-toolkit INFO     Invoking user script
Training Env:
{
    "ad

In [21]:
sklearn_estimator.latest_training_job.wait(logs="None")
artifact = sm_boto3.describe_training_job(
    TrainingJobName=sklearn_estimator.latest_training_job.name
)["ModelArtifacts"]["S3ModelArtifacts"]


2026-04-15 16:42:37 Starting - Preparing the instances for training
2026-04-15 16:42:37 Downloading - Downloading the training image
2026-04-15 16:42:37 Training - Training image download completed. Training in progress.
2026-04-15 16:42:37 Uploading - Uploading generated training model
2026-04-15 16:42:37 Completed - Training job completed


In [22]:
artifact

's3://sagemaker-us-east-1-889993502497/sklearn-mobile-price-classification-2026-04-15-16-40-05-113/output/model.tar.gz'

In [23]:
## deply the model to an endpoint

from sagemaker.sklearn.model import SKLearnModel
from time import gmtime, strftime

model_name=f"sklearn-mobile-price-classification-{strftime('%Y-%m-%d-%H-%M-%S', gmtime())}"
print(f"Deploying model to endpoint with name: {model_name}")

sklearn_model = SKLearnModel(
    name=model_name,
    model_data=artifact,
    role=ROLE_ARN,
    entry_point="script.py",
    framework_version=FRAMEWORK_VERSION
)

Deploying model to endpoint with name: sklearn-mobile-price-classification-2026-04-15-16-52-06


In [ ]:
## Endpoint deployment

from time import gmtime, strftime

# Rebuild sklearn_model if kernel state was reset.
if "sklearn_model" not in globals():
    required = ["artifact", "ROLE_ARN", "FRAMEWORK_VERSION"]
    missing = [name for name in required if name not in globals()]

    if not missing:
        from sagemaker.sklearn.model import SKLearnModel

        model_name = f"sklearn-mobile-price-classification-model-{strftime('%Y-%m-%d-%H-%M-%S', gmtime())}"
        sklearn_model = SKLearnModel(
            name=model_name,
            model_data=artifact,
            role=ROLE_ARN,
            entry_point="script.py",
            framework_version=FRAMEWORK_VERSION,
        )
        print("Recreated sklearn_model from artifact.")
    else:
        print(
            "Skipping deployment because prerequisites are missing: "
            + ", ".join(missing)
            + ". Run the earlier training/model-artifact cells first."
        )

if "sklearn_model" in globals():
    ENDPOINT_INSTANCE_TYPE_CANDIDATES = ["ml.m5.large", "ml.m5.xlarge", "ml.c5.xlarge"]
    end_point_name = f"sklearn-mobile-price-classification-endpoint-{strftime('%Y-%m-%d-%H-%M-%S', gmtime())}"

    predictor = None
    last_deploy_error = None

    for endpoint_instance_type in ENDPOINT_INSTANCE_TYPE_CANDIDATES:
        try:
            print(f"Deploying endpoint on {endpoint_instance_type}...")
            predictor = sklearn_model.deploy(
                initial_instance_count=1,
                instance_type=endpoint_instance_type,
                endpoint_name=end_point_name,
            )
            print(f"Endpoint deployed on {endpoint_instance_type}: {end_point_name}")
            break
        except Exception as exc:
            last_deploy_error = exc
            print(f"Deployment failed on {endpoint_instance_type}: {exc}")

    if predictor is None:
        raise RuntimeError("Endpoint deployment failed for all candidate instance types.") from last_deploy_error